# B07-01 — 실제 배지 조성과 성장 특성의 탐색적 통계 분석

**학습 목표**: B07-00 Clean v2에서 검증한 1,025개의 실제 배지 조성(`medium_profile_id`)을 기준으로 성분 분포, 조성별 성장 지표 분포, 단변량 상관, PCA, 라운드 차이 및 반복 측정의 영향을 조사한다.

**순서**: B07-00 Clean v2 완료 → **B07-01 실행** → 사용자 실행 결과 검토 → B07-01 상세 Word 해설서 → B07-02 예측 모델.

**엄격한 해석 규칙**
- 원본 `K`, `r`의 정확한 계산 정의는 미확인. 원본 평가 지표라는 사실만 명시한다.
- 조성별 집계값은 서로 다른 라운드의 측정 체계·조건이 섞일 수 있는 *탐색적 요약*이다. 상관은 인과관계가 아니다.
- 동일 조성의 반복 곡선은 독립적인 조건 1,025개로 환산되며, 13,608개를 독립 표본으로 취급하지 않는다.
- PCA는 성장 정보(`K`, `r`)를 **사용하지 않고** 배지 성분 44개만 이용한다. 표준화는 EDA용이고, 이후 ML에서는 매 학습 fold 안에서 새로 학습해야 한다.
- 개인 PC의 검증된 데이터 파일을 분석한다. 노트북을 배포하기 전 개인정보·로컬 경로가 포함된 실행 출력은 검토한다.

## 1. 환경 확인과 B07-00 승인 보고서 확인

In [ ]:
from pathlib import Path
import json, hashlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd.parent, cwd/'bioimage-ai') if (p/'results'/'bioculture'/'B07_00_ml_data_audit_v2').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('results/bioculture/B07_00_ml_data_audit_v2 폴더를 찾지 못했습니다.')
BASE=ROOT/'results'/'bioculture'
SRC=BASE/'B07_00_ml_data_audit_v2'
OUT=BASE/'B07_01_medium_growth_eda_v1'
OUT.mkdir(parents=True,exist_ok=True)
REPORT=SRC/'B07_00_validation_report.json'
CURVE=SRC/'ml_curve_level_audit_table.csv'
PROFILE=SRC/'ml_medium_profile_level_audit_table.csv'
FEATURE_AUDIT=SRC/'medium_feature_audit.csv'
for p in (REPORT,CURVE,PROFILE,FEATURE_AUDIT):
    if not p.exists(): raise FileNotFoundError(p)
report=json.loads(REPORT.read_text(encoding='utf-8'))
assert report.get('status')=='COMPOSITION_AUDIT_PASSED_NOT_MODEL_VALIDATION', report.get('status')
assert report.get('composition_consistent_by_medium_profile') is True
print('Project:', ROOT)
print('B07-00 status:', report['status'])
print('Audited profiles:',report['n_distinct_medium_profiles'])


## 2. 실제 데이터 로드 및 단위/조성 키 검증

In [ ]:
c=pd.read_csv(CURVE,encoding='utf-8-sig',low_memory=False,dtype={'Condition ID':'string','Assay ID':'string','Label':'string','medium_profile_id':'string'})
p=pd.read_csv(PROFILE,encoding='utf-8-sig',low_memory=False,dtype={'medium_profile_id':'string'})
fa=pd.read_csv(FEATURE_AUDIT,encoding='utf-8-sig')
features=fa['feature'].tolist()
assert len(features)==44 and all(x in c.columns and x in p.columns for x in features)
assert len(c)==report['n_curves']==13608
assert len(p)==report['n_distinct_medium_profiles']==1025
assert c['medium_profile_id'].notna().all() and p['medium_profile_id'].is_unique
assert set(c['medium_profile_id'])==set(p['medium_profile_id'])
assert not p[features].isna().any().any()
assert np.isfinite(p[features].to_numpy(dtype=float)).all()
# 조성 테이블에 저장된 n_curves 와 실제 곡선 행의 개수 대조
size=c.groupby('medium_profile_id').size()
assert p.set_index('medium_profile_id')['n_curves'].sort_index().equals(size.sort_index())
targets=['K','r','observed_max','growth_log_slope_per_h']
assert all(t in c.columns for t in targets)
print('Curves:',c.shape,'Profiles:',p.shape,'Features:',len(features))
print('Targets available:', {x:int(pd.to_numeric(c[x],errors='coerce').notna().sum()) for x in targets})


## 3. 조성 단위로 성장 목표 요약하기 — 반복 측정의 의미 보존

개별 곡선의 누락을 먼저 처리하고, 각 조성에서 **유효한 값만** 중앙값으로 집계한다. 반복 측정 개수(`n_*`)도 항상 함께 남긴다. 이 통계는 독립 실험 반복을 증명하지 않는다.

In [ ]:
summ=c.groupby('medium_profile_id',sort=True).agg(
    n_curves=('Label','size'),
    n_rounds=('round','nunique'),
    K_median=('K','median'), n_K=('K','count'),
    r_median=('r','median'), n_r=('r','count'),
    observed_max_median=('observed_max','median'), n_observed_max=('observed_max','count'),
    growth_log_slope_median=('growth_log_slope_per_h','median'), n_slope=('growth_log_slope_per_h','count'),
).reset_index()
summary=p[['medium_profile_id']+features].merge(summ,on='medium_profile_id',validate='one_to_one')
assert len(summary)==1025 and summary['n_curves'].sum()==13608
summary.to_csv(OUT/'eda_medium_profile_table.csv',index=False,encoding='utf-8-sig')
print('Profiles:',len(summary))
print(summary[['n_curves','n_rounds','n_K','n_r','n_observed_max','n_slope']].describe().to_string())


## 4. 성분 농도의 분포, 다중 척도, 0 농도

농도 0은 성분이 포함되지 않은 실험 조건을 나타낼 수 있으나, 원본 문헌상 의미를 확인하기 전에는 결측값으로 바꾸지 않는다. 성분별 값의 범위가 크게 달라 성분량을 직접 비교하는 그림에는 주의가 필요하다.

In [ ]:
feature_stats=pd.DataFrame([{
 'feature':x, 'n_profiles':len(summary), 'n_zero':int((summary[x]==0).sum()),
 'fraction_zero':float((summary[x]==0).mean()),
 'n_unique':int(summary[x].nunique()), 'median_mM':float(summary[x].median()),
 'q25_mM':float(summary[x].quantile(.25)), 'q75_mM':float(summary[x].quantile(.75)),
 'max_mM':float(summary[x].max())} for x in features]).sort_values('n_unique',ascending=False)
feature_stats.to_csv(OUT/'medium_component_distributions.csv',index=False,encoding='utf-8-sig')
print(feature_stats.head(12).to_string(index=False))
fig,ax=plt.subplots(figsize=(10,5))
ax.bar(range(min(20,len(feature_stats))),feature_stats['fraction_zero'].iloc[:20])
ax.set_xticks(range(min(20,len(feature_stats))))
ax.set_xticklabels([str(s).replace('\n',' ') for s in feature_stats['feature'].iloc[:20]],rotation=80,ha='right',fontsize=8)
ax.set_ylim(0,1);ax.set_ylabel('Fraction of medium profiles with 0 concentration');ax.set_title('Most-variable medium components: zero fractions')
fig.tight_layout();fig.savefig(OUT/'medium_component_zero_fraction.png',dpi=150);plt.show()


## 5. 원본 성장 지표의 분포 및 반복 관측 수

`K` 및 `r`은 원본 평가표의 수치이며 현재 생물학적 단위를 추정하지 않는다. 다양한 분포와 라운드 차이는 인과적 비교가 아니다.

In [ ]:
ycols=['K_median','r_median','observed_max_median','growth_log_slope_median']
coverage=pd.DataFrame([{'target':x,'n_valid_profiles':int(summary[x].notna().sum()),'n_missing_profiles':int(summary[x].isna().sum()),
                        'median':float(summary[x].median()) if summary[x].notna().any() else None,
                        'q25':float(summary[x].quantile(.25)) if summary[x].notna().any() else None,
                        'q75':float(summary[x].quantile(.75)) if summary[x].notna().any() else None} for x in ycols])
coverage.to_csv(OUT/'target_coverage_and_distribution.csv',index=False,encoding='utf-8-sig')
print(coverage.to_string(index=False))
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,name in zip(axes,['K_median','r_median']):
    ax.hist(summary[name].dropna(),bins=35)
    ax.set_title(f'Profile-median {name}');ax.set_xlabel('Original dataset metric (definition unverified)');ax.set_ylabel('Number of medium profiles')
fig.tight_layout();fig.savefig(OUT/'profile_target_histograms.png',dpi=150);plt.show()


## 6. 단변량 상관분석 — 유효 조성 수와 다중 비교 한계

스피어만 상관은 단조 연관성을 탐색한다. 44개 성분 × 목표 여러 개를 동시에 비교하므로 상위 순위나 작은 p 값만으로 인과성·유의성을 확정하지 않는다. 농도 조합의 공변화와 라운드 교란이 존재할 수 있다.

In [ ]:
corrows=[]
for target in ycols:
    for x in features:
        pair=summary[[x,target]].replace([np.inf,-np.inf],np.nan).dropna()
        if len(pair)>=10 and pair[x].nunique()>1 and pair[target].nunique()>1:
            rho=float(pair[x].corr(pair[target],method='spearman'))
        else: rho=np.nan
        corrows.append({'target':target,'feature':x,'n_profiles':len(pair),'spearman_rho':rho})
corr=pd.DataFrame(corrows)
corr['absolute_rho']=corr['spearman_rho'].abs()
corr.to_csv(OUT/'univariate_spearman_screen.csv',index=False,encoding='utf-8-sig')
for t in ['K_median','r_median']:
    print('\n',t,'— highest absolute correlations (exploratory):')
    print(corr[corr.target.eq(t)].sort_values('absolute_rho',ascending=False).head(10).to_string(index=False))


## 7. PCA — 성장 목표를 제외한 배지 조성 구조

44개 성분만 StandardScaler로 평균 0, 표준편차 1로 표준화한 후 PCA를 적용한다. 전체 데이터에서 PCA를 적합하는 것은 **탐색적 시각화에 한정**된다. B07-02 모델 검증에서는 훈련 자료 안에서만 전처리를 적합한다.

In [ ]:
X=summary[features].to_numpy(dtype=float)
scaler=StandardScaler()
Z=scaler.fit_transform(X)
pca=PCA(n_components=2)
PC=pca.fit_transform(Z)
assert PC.shape==(1025,2)
scores=pd.DataFrame({'medium_profile_id':summary['medium_profile_id'],'PC1':PC[:,0],'PC2':PC[:,1],
                     'K_median':summary['K_median'],'r_median':summary['r_median'],'n_curves':summary['n_curves']})
scores.to_csv(OUT/'medium_PCA_scores.csv',index=False,encoding='utf-8-sig')
loads=pd.DataFrame({'feature':features,'PC1_loading':pca.components_[0], 'PC2_loading':pca.components_[1]})
loads.to_csv(OUT/'medium_PCA_loadings.csv',index=False,encoding='utf-8-sig')
print('PCA explained variance ratio:',pca.explained_variance_ratio_)
fig,ax=plt.subplots(figsize=(7,5)); pts=ax.scatter(scores.PC1,scores.PC2,c=scores.K_median,cmap='viridis',s=16,alpha=.75)
ax.set_xlabel('PC1 (medium composition only)');ax.set_ylabel('PC2');ax.set_title('Medium composition PCA, color = original K median')
fig.colorbar(pts,ax=ax,label='Original K (definition unverified)')
fig.tight_layout();fig.savefig(OUT/'medium_composition_PCA.png',dpi=150);plt.show()


## 8. 라운드별 측정 편차 및 조건 구성 점검

라운드 사이의 성장 지표 차이는 조성 및 장비/실험 운영의 차이와 뒤섞여 있을 수 있다. 동일 조성의 여러 라운드 관측이 있는지 별도로 기록하며 원인 효과로 해석하지 않는다.

In [ ]:
round_summary=c.groupby('round',dropna=False).agg(n_curves=('Label','size'),n_profiles=('medium_profile_id','nunique'),
 K_median=('K','median'),r_median=('r','median'),n_K=('K','count'),n_r=('r','count')).reset_index()
round_summary.to_csv(OUT/'round_level_target_audit.csv',index=False,encoding='utf-8-sig')
repeat=summary[['medium_profile_id','n_curves','n_rounds']].copy()
repeat.to_csv(OUT/'medium_replication_audit.csv',index=False,encoding='utf-8-sig')
print(round_summary.to_string(index=False))
print('Profiles present in >1 round:',int((repeat.n_rounds>1).sum()))
print('Curves/profile distribution:',repeat.n_curves.describe().to_dict())


## 9. 저장 결과 재검증 및 입력·출력 SHA256

모든 주요 결과는 `results/bioculture/B07_01_medium_growth_eda_v1`에 기록된다. 원본 파일을 덮어쓰지 않는다.

In [ ]:
reloaded=pd.read_csv(OUT/'eda_medium_profile_table.csv',encoding='utf-8-sig',low_memory=False)
assert len(reloaded)==1025 and reloaded.medium_profile_id.is_unique
assert int(reloaded.n_curves.sum())==13608
assert set(reloaded.medium_profile_id)==set(summary.medium_profile_id)
assert len(pd.read_csv(OUT/'univariate_spearman_screen.csv',encoding='utf-8-sig'))==len(features)*len(ycols)
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
    return h.hexdigest()
result={'status':'EDA_COMPLETED_NOT_CAUSAL_NOT_MODEL_VALIDATION',
        'n_curves':len(c),'n_medium_profiles':len(summary),'n_features':len(features),
        'n_profile_targets':{x:int(summary[x].notna().sum()) for x in ycols},
        'pca_explained_variance_ratio':pca.explained_variance_ratio_.tolist(),
        'n_profiles_multiple_rounds':int((summary.n_rounds>1).sum()),
        'data_split_executed':False,'predictive_model_fitted':False,
        'caveats':['Association not causation','K/r definitions unverified','Replicates not assumed independent','Round confounding possible','PCA uses all data ONLY for unsupervised EDA']}
(OUT/'B07_01_validation_report.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
manifest=[{'role':'input','file':str(f.relative_to(ROOT)),'sha256':sha256(f)} for f in (REPORT,CURVE,PROFILE,FEATURE_AUDIT)]
manifest +=[{'role':'output','file':str(f.relative_to(ROOT)),'sha256':sha256(f)} for f in sorted(OUT.iterdir()) if f.is_file() and f.name!='B07_01_SHA256_manifest.csv']
pd.DataFrame(manifest).to_csv(OUT/'B07_01_SHA256_manifest.csv',index=False,encoding='utf-8-sig')
print(json.dumps(result,ensure_ascii=False,indent=2))
print('Output folder:',OUT,'saved files:',len(list(OUT.iterdir())))


## 연습문제 — 해설과 Python 정답은 별도 Word 해설서 수록

1. `n_curves=588`인 조성 하나가 `n_curves=6`인 조성보다 상관분석에 과도한 영향력을 행사하지 않도록, 본 Notebook은 어떤 분석 단위를 선택했는가? 이 방식에도 남는 제약은 무엇인가?
2. `K_median` 및 `r_median`이 결측인 조성 수를 계산하고, 결측을 0으로 바꾸면 안 되는 이유를 설명하라.
3. 스피어만 상관 상위 성분 5개를 확인하고, 그 순위를 인과효과로 읽을 수 없는 이유를 2가지 기술하라.
4. PCA에서 `K`나 `r`을 입력에 포함하지 않은 이유와 이후 교차검증에서 StandardScaler를 다시 적합해야 하는 이유를 설명하라.
5. 여러 라운드에 걸쳐 측정된 배지 조성을 찾고, 라운드 간 성장값이 차이 날 때 가능한 해석과 확인할 추가 자료를 제시하라.

**다음 단계:** 실행 완료 Notebook을 업로드해 주세요. 실행 결과를 검토하고 필요한 경우 수정한 뒤 상세 Word 해설서를 작성합니다. 이후 B07-02 모델링으로 진행합니다.